# Part 4: RAG + LLM Integration

This notebook implements:
- Free LLM integration with API management
- RBAC-aware retrieval pipeline
- Prompt templates with context augmentation
- Source citation and document attribution
- Confidence scoring based on retrieval relevance
- End-to-end test cases

In [1]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Imports ready")

✅ Imports ready


In [2]:
print("🔐 Loading users, roles, and chunk corpus...")

ROLE_HIERARCHY = {
    "admin": 100,
    "ceo": 100,
    "leadership": 90,
    "engineering_lead": 80,
    "tech_lead": 80,
    "finance_lead": 80,
    "hr_lead": 80,
    "sales": 80,
    "engineering": 60,
    "finance": 60,
    "hr": 60,
    "marketing": 60,
    "all_employees": 40,
}

DEPARTMENT_ACCESS = {
    "engineering": {"admin": True, "ceo": True, "leadership": True, "engineering_lead": True, "tech_lead": True, "engineering": True},
    "finance": {"admin": True, "ceo": True, "leadership": True, "finance_lead": True, "finance": True},
    "hr": {"admin": True, "ceo": True, "leadership": True, "hr_lead": True, "hr": True},
    "marketing": {"admin": True, "ceo": True, "leadership": True, "sales": True, "marketing": True, "all_employees": True},
    "general": {"admin": True, "ceo": True, "leadership": True, "all_employees": True},
}

# Sample accounts for notebook auth flow
USER_STORE = {
    "admin_user": {"password": "AdminPass123!", "role": "admin"},
    "finance_user": {"password": "FinancePass123!", "role": "finance"},
    "hr_user": {"password": "HRPass123!", "role": "hr"},
    "engineering_user": {"password": "EngPass123!", "role": "engineering"},
    "employee_user": {"password": "EmployeePass123!", "role": "all_employees"},
}


def resolve_chunks_path() -> Path:
    candidates = [
        Path.cwd() / "artifacts" / "module2" / "chunks" / "cleaned_formatted_chunks.csv",
        Path.cwd().parent / "artifacts" / "module2" / "chunks" / "cleaned_formatted_chunks.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find cleaned_formatted_chunks.csv")


chunks_path = resolve_chunks_path()
chunks_df = pd.read_csv(chunks_path)
chunks = chunks_df.to_dict("records")

for c in chunks:
    roles = c.get("accessible_roles", "")
    if isinstance(roles, str):
        c["accessible_roles"] = [r.strip() for r in roles.split(",") if r.strip()]

print(f"✅ Loaded {len(chunks)} chunks from {chunks_path}")

🔐 Loading users, roles, and chunk corpus...
✅ Loaded 311 chunks from d:\Projects\RBAC\artifacts\module2\chunks\cleaned_formatted_chunks.csv


In [3]:
print("🧠 Building RBAC + retrieval utilities...")


def normalize_query(query: str) -> str:
    query = query.lower()
    query = re.sub(r"[^a-z0-9\s]", " ", query)
    query = re.sub(r"\s+", " ", query).strip()
    return query


def authenticate_user(username: str, password: str) -> Dict:
    user = USER_STORE.get(username)
    if not user or user["password"] != password:
        raise ValueError("Authentication failed")
    return {"username": username, "role": user["role"], "access_level": ROLE_HIERARCHY.get(user["role"], 0)}


def can_access_department(role: str, department: str) -> bool:
    dept_perms = DEPARTMENT_ACCESS.get(str(department).lower(), {})
    return dept_perms.get(role, False)


def filter_chunks_by_role(role: str, records: List[Dict]) -> List[Dict]:
    allowed = []
    for c in records:
        dept = str(c.get("department", "")).lower()
        if not can_access_department(role, dept):
            continue

        # Privileged override
        if ROLE_HIERARCHY.get(role, 0) >= 90:
            allowed.append(c)
            continue

        allowed_roles = c.get("accessible_roles", [])
        if not allowed_roles or role in allowed_roles:
            allowed.append(c)

    return allowed


def retrieve_relevant_chunks(query: str, candidate_chunks: List[Dict], top_k: int = 5) -> List[Dict]:
    if not candidate_chunks:
        return []

    q = normalize_query(query)
    docs = [normalize_query(str(c.get("content", ""))) for c in candidate_chunks]

    vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
    doc_matrix = vectorizer.fit_transform(docs)
    q_vec = vectorizer.transform([q])

    sims = cosine_similarity(q_vec, doc_matrix)[0]
    ranked_idx = sims.argsort()[::-1][:top_k]

    results = []
    for idx in ranked_idx:
        item = dict(candidate_chunks[idx])
        item["retrieval_score"] = float(sims[idx])
        results.append(item)

    return results


print("✅ Utilities ready")

🧠 Building RBAC + retrieval utilities...
✅ Utilities ready


In [4]:
print("📝 Defining prompt templates and augmentation...")

SYSTEM_PROMPT = """You are FinSolve Assistant.
Answer using ONLY the provided context.
If the answer is not in context, say you do not have enough information.
Always cite sources as [chunk_id | source_document | department].
Keep answers concise and factual.
"""

CONTEXT_TEMPLATE = """User Role: {role}
Question: {question}

Retrieved Context:
{context_blocks}

Instructions:
1) Use only retrieved context.
2) Provide a direct answer.
3) Add citations for each key claim.
"""


def build_context_blocks(retrieved_chunks: List[Dict]) -> str:
    blocks = []
    for i, c in enumerate(retrieved_chunks, start=1):
        block = (
            f"[{i}] chunk_id={c.get('chunk_id')} | doc={c.get('source_document')} | dept={c.get('department')} | score={c.get('retrieval_score'):.4f}\n"
            f"{str(c.get('content', ''))[:1200]}"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


def build_augmented_prompt(question: str, role: str, retrieved_chunks: List[Dict]) -> Tuple[str, str]:
    context_blocks = build_context_blocks(retrieved_chunks)
    user_prompt = CONTEXT_TEMPLATE.format(role=role, question=question, context_blocks=context_blocks)
    return SYSTEM_PROMPT, user_prompt


print("✅ Prompt templates ready")

📝 Defining prompt templates and augmentation...
✅ Prompt templates ready


In [5]:
print("🤖 Initializing LLM API management module...")

@dataclass
class LLMConfig:
    provider: str = "mock"  # "openai", "huggingface", or "mock"
    openai_api_key_env: str = "OPENAI_API_KEY"
    openai_model: str = "gpt-4o-mini"
    hf_api_key_env: str = "HF_API_TOKEN"
    hf_model: str = "google/flan-t5-base"
    timeout_seconds: int = 45


class LLMClient:
    def __init__(self, config: LLMConfig):
        self.config = config

    def _call_openai(self, system_prompt: str, user_prompt: str) -> str:
        api_key = os.getenv(self.config.openai_api_key_env, "")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not set")

        url = "https://api.openai.com/v1/chat/completions"
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        payload = {
            "model": self.config.openai_model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": 0.1,
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=self.config.timeout_seconds)
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"].strip()

    def _call_huggingface(self, system_prompt: str, user_prompt: str) -> str:
        token = os.getenv(self.config.hf_api_key_env, "")
        if not token:
            raise ValueError("HF_API_TOKEN not set")

        url = f"https://api-inference.huggingface.co/models/{self.config.hf_model}"
        headers = {"Authorization": f"Bearer {token}"}
        prompt = f"System: {system_prompt}\n\nUser: {user_prompt}\n\nAssistant:"
        payload = {"inputs": prompt, "parameters": {"max_new_tokens": 250, "temperature": 0.1}}

        resp = requests.post(url, headers=headers, json=payload, timeout=self.config.timeout_seconds)
        resp.raise_for_status()
        data = resp.json()

        if isinstance(data, list) and data and "generated_text" in data[0]:
            return str(data[0]["generated_text"]).strip()
        if isinstance(data, dict) and "generated_text" in data:
            return str(data["generated_text"]).strip()
        return str(data)

    def _mock_grounded_answer(self, question: str, retrieved_chunks: List[Dict]) -> str:
        if not retrieved_chunks:
            return "I do not have enough information in the accessible documents to answer this question."

        top = retrieved_chunks[0]
        preview = str(top.get("content", "")).replace("\n", " ")[:450]
        citation = f"[{top.get('chunk_id')} | {top.get('source_document')} | {top.get('department')}]"
        return f"Based on the top retrieved context, {preview} ... {citation}"

    def generate(self, question: str, system_prompt: str, user_prompt: str, retrieved_chunks: List[Dict]) -> str:
        provider = self.config.provider.lower()

        if provider == "openai":
            try:
                return self._call_openai(system_prompt, user_prompt)
            except Exception as exc:
                return f"[OpenAI fallback] {self._mock_grounded_answer(question, retrieved_chunks)}\nReason: {exc}"

        if provider == "huggingface":
            try:
                return self._call_huggingface(system_prompt, user_prompt)
            except Exception as exc:
                return f"[HF fallback] {self._mock_grounded_answer(question, retrieved_chunks)}\nReason: {exc}"

        return self._mock_grounded_answer(question, retrieved_chunks)


llm_client = LLMClient(LLMConfig(provider="mock"))
print("✅ LLM module initialized (provider=mock)")
print("   Switch to 'openai' or 'huggingface' in LLMConfig when API keys are available.")

🤖 Initializing LLM API management module...
✅ LLM module initialized (provider=mock)
   Switch to 'openai' or 'huggingface' in LLMConfig when API keys are available.


In [6]:
print("🧩 Implementing end-to-end RAG pipeline...")


def compute_confidence(retrieved_chunks: List[Dict]) -> float:
    if not retrieved_chunks:
        return 0.0

    scores = [max(0.0, min(1.0, float(c.get("retrieval_score", 0.0)))) for c in retrieved_chunks]
    top = scores[0]
    mean_top3 = sum(scores[:3]) / min(3, len(scores))
    confidence = 0.65 * top + 0.35 * mean_top3
    return round(confidence, 4)


def build_citations(retrieved_chunks: List[Dict]) -> List[Dict]:
    citations = []
    for c in retrieved_chunks:
        citations.append(
            {
                "chunk_id": c.get("chunk_id"),
                "source_document": c.get("source_document"),
                "department": c.get("department"),
                "retrieval_score": round(float(c.get("retrieval_score", 0.0)), 4),
            }
        )
    return citations


def rag_query(
    username: str,
    password: str,
    query: str,
    top_k: int = 5,
    llm: LLMClient = llm_client,
) -> Dict:
    # 1) authenticate
    user = authenticate_user(username, password)
    role = user["role"]

    # 2) filter by role
    allowed_chunks = filter_chunks_by_role(role, chunks)

    # 3) retrieve relevant chunks
    retrieved = retrieve_relevant_chunks(query, allowed_chunks, top_k=top_k)

    # 4) augment prompt
    system_prompt, user_prompt = build_augmented_prompt(query, role, retrieved)

    # 5) generate response
    answer = llm.generate(query, system_prompt, user_prompt, retrieved)

    # 6) source attribution + confidence
    citations = build_citations(retrieved)
    confidence = compute_confidence(retrieved)

    return {
        "user": username,
        "role": role,
        "query": query,
        "answer": answer,
        "confidence": confidence,
        "sources": citations,
        "retrieved_count": len(retrieved),
    }


print("✅ RAG pipeline ready")

🧩 Implementing end-to-end RAG pipeline...
✅ RAG pipeline ready


In [7]:
print("🧪 Running RAG functionality test cases...\n")

test_cases = [
    {
        "name": "Finance user finance question",
        "username": "finance_user",
        "password": "FinancePass123!",
        "query": "What were the quarterly financial results and revenue trend?",
        "expect_department_contains": "finance",
        "expect_forbidden_department": "hr",
    },
    {
        "name": "HR user policy question",
        "username": "hr_user",
        "password": "HRPass123!",
        "query": "What are the employee leave and policy guidelines?",
        "expect_department_contains": "hr",
        "expect_forbidden_department": "finance",
    },
    {
        "name": "General employee cannot see finance",
        "username": "employee_user",
        "password": "EmployeePass123!",
        "query": "Show me company announcements and benefits.",
        "expect_forbidden_department": "finance",
    },
]

results = []
for tc in test_cases:
    out = rag_query(tc["username"], tc["password"], tc["query"], top_k=5)
    source_depts = [s["department"] for s in out["sources"]]

    passed = True
    reasons = []

    if tc.get("expect_department_contains") and tc["expect_department_contains"] not in source_depts:
        passed = False
        reasons.append(f"Expected department '{tc['expect_department_contains']}' not in sources")

    if tc.get("expect_forbidden_department") and tc["expect_forbidden_department"] in source_depts:
        passed = False
        reasons.append(f"Forbidden department '{tc['expect_forbidden_department']}' appeared in sources")

    if out["retrieved_count"] == 0:
        passed = False
        reasons.append("No chunks retrieved")

    if out["confidence"] < 0:
        passed = False
        reasons.append("Invalid confidence score")

    results.append(
        {
            "test": tc["name"],
            "passed": passed,
            "retrieved_count": out["retrieved_count"],
            "confidence": out["confidence"],
            "source_departments": sorted(set(source_depts)),
            "notes": "; ".join(reasons) if reasons else "ok",
            "answer_preview": out["answer"][:180].replace("\n", " "),
        }
    )

results_df = pd.DataFrame(results)
print(results_df[["test", "passed", "retrieved_count", "confidence", "source_departments", "notes"]].to_string(index=False))

report_dir = Path.cwd() / "artifacts" / "module2" / "reports"
if not report_dir.exists():
    report_dir = Path.cwd().parent / "artifacts" / "module2" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

csv_path = report_dir / "rag_test_results.csv"
md_path = report_dir / "rag_test_report.md"
results_df.to_csv(csv_path, index=False)

passed_n = int(results_df["passed"].sum())
total_n = len(results_df)

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# RAG Functionality Test Report\n\n")
    f.write(f"Passed: {passed_n}/{total_n}\n\n")
    f.write("## Results\n\n")
    f.write(results_df.to_string(index=False))

print(f"\n✅ Saved test results: {csv_path}")
print(f"✅ Saved test report: {md_path}")

🧪 Running RAG functionality test cases...

                               test  passed  retrieved_count  confidence source_departments notes
      Finance user finance question    True                5      0.1436          [finance]    ok
            HR user policy question    True                5      0.0000               [hr]    ok
General employee cannot see finance    True                5      0.1408          [general]    ok

✅ Saved test results: d:\Projects\RBAC\artifacts\module2\reports\rag_test_results.csv
✅ Saved test report: d:\Projects\RBAC\artifacts\module2\reports\rag_test_report.md
